# Local LLM Fine-Tuning Studio — Colab runner

Runs the full pipeline from this repo on a Colab GPU runtime: data prep → QLoRA/LoRA training → evaluation → inference.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or better).

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

Set `REPO_URL` if you forked it, or upload the repo as a zip instead and skip this cell.

In [ ]:
REPO_URL = "https://github.com/abduroshyd/local-llm-finetune-unsloth.git"

import os
if not os.path.exists("local-llm-finetune-unsloth"):
    !git clone $REPO_URL
%cd local-llm-finetune-unsloth

## 3. Install dependencies

Colab already ships torch; we only add Unsloth and the rest of the stack.

In [ ]:
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -r requirements.txt

## 4. (Optional) Hugging Face login

Required only if you're training `llama3.2` (gated checkpoint). Not needed for `qwen2.5`.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Prepare the dataset

The repo ships a 43-example seed dataset at `data/processed/train.jsonl` / `val.jsonl`, ready to use as-is.

To train on a larger public dataset instead, uncomment and run the conversion pipeline below (see `docs/DATASET_GUIDE.md` for source options).

In [ ]:
# !python scripts/convert_to_alpaca.py --source dolly --output data/raw/dolly_alpaca.jsonl
# !python scripts/clean_dataset.py --input data/raw/dolly_alpaca.jsonl --output data/raw/dolly_clean.jsonl
# !python scripts/split_dataset.py --input data/raw/dolly_clean.jsonl \
#     --train-output data/processed/train.jsonl --val-output data/processed/val.jsonl --val-ratio 0.1

!wc -l data/processed/train.jsonl data/processed/val.jsonl

## 6. Fine-tune

`--mode qlora` fits comfortably on a free-tier T4 (16GB). Switch `--model` between `llama3.2` and `qwen2.5`, and set `--adapter-name` to whatever you want the saved adapter directory to be called.

In [ ]:
!python training/train.py --mode qlora --model llama3.2 --adapter-name llama3.2-qlora

## 7. Evaluate: base vs. fine-tuned

In [ ]:
!python evaluation/evaluate.py --adapter llama3.2-qlora

## 8. Try it out

In [ ]:
!python inference/generate.py --adapter llama3.2-qlora --instruction "Explain LoRA in one paragraph."

## 9. Download the trained adapter + reports

Zips `models/adapters/<name>/` (adapter weights + `training_stats.json`) and `evaluation/results/` so you can copy them back into your local repo for the README's Results section and the Streamlit app's Training Stats tab.

In [ ]:
ADAPTER_NAME = "llama3.2-qlora"
!zip -r /content/{ADAPTER_NAME}_results.zip models/adapters/{ADAPTER_NAME} evaluation/results

from google.colab import files
files.download(f"/content/{ADAPTER_NAME}_results.zip")